# 02 · Protein–ligand pocket mapping

Extract residues close to the configured ligand in pre-prepared complex PDB files. Structural helper functions are collaborator-provided (PD Dr. Behnam Yousefi). This stage requires PyMOL, PLIP, RDKit and Biopython as imported by the helper.

## Configuration

Use `config.local.json` as described in [the workflow guide](../docs/WORKFLOW.md). Inputs must be authorized for local analysis. Outputs are cleared in version control; no research result is asserted by this notebook.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis/interactions.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Start Jupyter from this repository')
sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analysis.notebook_setup import load_config, load_manifest
config = load_config()
out = config['output_dir']


## Load explicit compound/pose manifest

In [ ]:
manifest = load_manifest(config)
manifest[['compound_id', 'pose_id', 'group']].head()

## Extract nearby residues without temporary-file deletion

In [ ]:
sys.path.insert(0, str(root / 'notebooks'))
from src.generate_graph_new import protein_pocket
frames = []
for row in manifest.itertuples(index=False):
    pocket = protein_pocket(row.complex_path, ligand_id=config['ligand_id'],
                            threshold=float(config['pocket_distance_angstrom']))
    if pocket.empty:
        raise ValueError(f'No pocket found for compound {row.compound_id}, pose {row.pose_id}; verify ligand ID')
    pocket = pocket.assign(compound_id=row.compound_id, pose_id=row.pose_id, group=row.group)
    frames.append(pocket)
pockets = pd.concat(frames, ignore_index=True)
pockets.to_csv(out / 'pocket_residues.csv', index=False)
pockets.head()